In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# Project root
# This notebook is located inside PD-HAND/spiral/
PROJECT_ROOT = Path.cwd().parent

# Dataset directory
DATA_DIR = PROJECT_ROOT / "dataset"

# Original HandPD dataset paths
pd_path_original = DATA_DIR / "raw" / "spiral" / "parkinson" / "SpiralPatients"
healthy_path_original = DATA_DIR / "raw" / "spiral" / "healthy" / "SpiralControl"

# Camera-style dataset paths
pd_path_camera = DATA_DIR / "raw_camera" / "spiral_camera" / "parkinson" / "SpiralPatients"
healthy_path_camera = DATA_DIR / "raw_camera" / "spiral_camera" / "healthy" / "SpiralControl"

# Real captured handwriting sample paths
pd_path_real = DATA_DIR / "real" / "spiral" / "parkinson"
healthy_path_real = DATA_DIR / "real" / "spiral" / "healthy"

# Combine all three data sources
pd_path = [pd_path_original, pd_path_camera, pd_path_real]
healthy_path = [healthy_path_original, healthy_path_camera, healthy_path_real]

# Load image filenames
pd_files = []
for folder in pd_path:
    pd_files.extend([
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

healthy_files = []
for folder in healthy_path:
    healthy_files.extend([
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

print("PD images:", len(pd_files))
print("Healthy images:", len(healthy_files))

In [ ]:
def preprocess_image(image_path, size=(128, 128)):
    image = cv2.imread(image_path)

    if image is None:
        raise ValueError(f"Unable to read image from path: {image_path}")
    
    original = image.copy()
    
    resized = cv2.resize(image, size)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

    # NEW: CLAHE enhancement
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )
    gray = clahe.apply(gray)

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    _, thresh = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    
    return original, gray, thresh

In [ ]:
from skimage.feature import hog, local_binary_pattern

def extract_features(thresh_img):
    features = {}

    # ── original shape features (keep all) ──────────────────────────────
    ink_pixels  = np.sum(thresh_img > 0)
    total_pixels = thresh_img.size
    ink_ratio   = ink_pixels / total_pixels

    features["ink_pixels"] = int(ink_pixels)
    features["ink_ratio"]  = float(ink_ratio)

    contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    features["num_contours"] = len(contours)

    if contours:
        largest_contour  = max(contours, key=cv2.contourArea)
        contour_area     = cv2.contourArea(largest_contour)
        contour_perimeter= cv2.arcLength(largest_contour, True)
        x, y, w, h       = cv2.boundingRect(largest_contour)
        aspect_ratio     = w / h if h != 0 else 0
        M = cv2.moments(largest_contour)
        if M["m00"] != 0:
            centroid_x = M["m10"] / M["m00"]
            centroid_y = M["m01"] / M["m00"]
        else:
            centroid_x, centroid_y = 0, 0
        rect_area = w * h
        extent    = contour_area / rect_area if rect_area != 0 else 0
        hull      = cv2.convexHull(largest_contour)
        hull_area = cv2.contourArea(hull)
        solidity  = contour_area / hull_area if hull_area != 0 else 0
    else:
        contour_area = contour_perimeter = 0
        w = h = 0
        aspect_ratio = centroid_x = centroid_y = extent = solidity = 0

    features["contour_area"]        = float(contour_area)
    features["contour_perimeter"]   = float(contour_perimeter)
    features["bounding_box_width"]  = int(w)
    features["bounding_box_height"] = int(h)
    features["aspect_ratio"]        = float(aspect_ratio)
    features["centroid_x"]          = float(centroid_x)
    features["centroid_y"]          = float(centroid_y)
    features["extent"]              = float(extent)
    features["solidity"]            = float(solidity)

    # NEW 1: HOG — captures stroke edge direction & tremor patterns 
    hog_features = hog(
        thresh_img,
        orientations=8,
        pixels_per_cell=(16, 16),
        cells_per_block=(1, 1),
        visualize=False
    )
    for i, val in enumerate(hog_features):
        features[f"hog_{i}"] = float(val)  # adds 512 features

    # NEW 2: LBP — captures local texture roughness of strokes
    lbp = local_binary_pattern(thresh_img, P=8, R=1, method="uniform")
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
    for i, val in enumerate(lbp_hist):
        features[f"lbp_{i}"] = float(val)  # adds 10 features

    # NEW 3: Hu moments — rotation-invariant spiral shape descriptors 
    moments    = cv2.moments(thresh_img)
    hu_moments = cv2.HuMoments(moments).flatten()
    for i, val in enumerate(hu_moments):
        features[f"hu_{i}"] = float(np.sign(val) * np.log1p(abs(val)))  # adds 7 features

    return features  # 12 + 512 + 10 + 7 = 541 features total

In [ ]:
def simulate_plain_paper(thresh_img):
    """Generate plain-paper style variants from a thresholded dataset image."""
    variants = []
    _, dark_only = cv2.threshold(thresh_img, 127, 255, cv2.THRESH_BINARY)
    variants.append(dark_only)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    dilated = cv2.dilate(dark_only, kernel, iterations=1)
    variants.append(dilated)
    eroded = cv2.erode(dark_only, kernel, iterations=1)
    variants.append(eroded)
    return variants

data = []

# --- Original dataset images (PD) ---
for folder in pd_path:
    for file in os.listdir(folder):
        if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        path = os.path.join(folder, file)
        _, _, thresh = preprocess_image(path)  # ✅ INSIDE THE LOOP!
        if thresh is None:
            continue
        features = extract_features(thresh)
        features["label"] = 1
        features["filename"] = file
        data.append(features)

        for aug_thresh in simulate_plain_paper(thresh):
            features_aug = extract_features(aug_thresh)
            features_aug["label"] = 1
            features_aug["filename"] = file + "_aug"
            data.append(features_aug)

# --- Original dataset images (Healthy) ---
for folder in healthy_path:
    for file in os.listdir(folder):
        if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        path = os.path.join(folder, file)
        _, _, thresh = preprocess_image(path)  # ✅ INSIDE THE LOOP!
        if thresh is None:
            continue
        features = extract_features(thresh)
        features["label"] = 0
        features["filename"] = file
        data.append(features)

        for aug_thresh in simulate_plain_paper(thresh):
            features_aug = extract_features(aug_thresh)
            features_aug["label"] = 0
            features_aug["filename"] = file + "_aug"
            data.append(features_aug)

# ===== BALANCING CODE (NEW!) =====
from sklearn.utils import resample

# Convert data list to temporary DataFrame to balance
df_temp = pd.DataFrame(data)

# Count samples per class
pd_count = len(df_temp[df_temp['label'] == 1])
healthy_count = len(df_temp[df_temp['label'] == 0])

print(f"Before balancing: PD={pd_count}, Healthy={healthy_count}, Ratio={pd_count/healthy_count:.2f}x")

# Undersample PD to match Healthy
if pd_count > healthy_count:
    df_pd = df_temp[df_temp['label'] == 1]
    df_healthy = df_temp[df_temp['label'] == 0]
    
    # Random sample without replacement
    df_pd_balanced = resample(
        df_pd,
        n_samples=healthy_count,
        random_state=42,
        replace=False  # Don't repeat samples
    )
    
    df = pd.concat([df_pd_balanced, df_healthy], ignore_index=True)
    df = df.sample(frac=1, random_state=42)  # Shuffle
else:
    df = df_temp

print(f"After balancing: PD={len(df[df['label']==1])}, Healthy={len(df[df['label']==0])}, Ratio={len(df[df['label']==1])/len(df[df['label']==0]):.2f}x")
# ===== END BALANCING CODE =====

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

feature_output_path = OUTPUT_DIR / "spiral_features_final.csv"
df.to_csv(feature_output_path, index=False)

print(f"Total samples (original + augmented): {len(df)}")
df.head()

In [ ]:
print("=== Dataset Overview ===")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nLabel Distribution:")
print(df["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["label", "filename"])
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== Train-Test Split ===")
print("Train:", X_train.shape)
print("Test:", X_test.shape)

# Models Baseline and Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight='balanced'
        ))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        max_features='sqrt',
        min_samples_leaf=1,
        min_samples_split=2,
        random_state=42,
        class_weight='balanced'
    ),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            probability=True,
            random_state=42,
            class_weight='balanced'
        ))
    ])
}

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
results_path = OUTPUT_DIR / "spiral_model_baseline_results.csv"
results_df.to_csv(results_path, index=False)

print("Results saved to:", results_path)

# Cross-validation (CV)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1")
    
    cv_results.append({
        "Model": name,
        "CV F1 Mean": scores.mean(),
        "CV F1 Std": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)
cv_results_df

In [ ]:
from sklearn.model_selection import GridSearchCV

# Tune SVM specifically for spiral
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(
        probability=True,
        random_state=42,
        class_weight='balanced'
    ))
])

param_grid = {
    'model__C'     : [0.1, 1, 10, 100],
    'model__gamma' : ['scale', 'auto', 0.001, 0.01],
    'model__kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(
    svm_pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'Best SVM parameters : {grid_search.best_params_}')
print(f'Best CV F1          : {grid_search.best_score_:.4f}')

# Evaluate tuned SVM
y_pred_tuned = grid_search.predict(X_test)
print(f'Tuned SVM Accuracy  : {accuracy_score(y_test, y_pred_tuned)*100:.2f}%')
print(f'Tuned SVM F1        : {f1_score(y_test, y_pred_tuned):.4f}')
print(f'Confusion Matrix:\n{confusion_matrix(y_test, y_pred_tuned)}')

# Final Overall Model Comparison Results

Model Baseline Results

In [ ]:
results_df

Cross-validation Results

In [ ]:
cv_results_df

Overall Model Results Table

In [ ]:
spiral_final_results_df = pd.merge(results_df, cv_results_df, on="Model")

spiral_final_results_df = spiral_final_results_df[
    ["Model", "Accuracy", "Precision", "Recall", "F1-score",
     "CV F1 Mean", "CV F1 Std", "TN", "FP", "FN", "TP"]
].round(4)

spiral_final_results_df

## Final Selected Spiral Model for Deployment
This model uses:
- Otsu threshold preprocessing  
- All 12 extracted image-based features  
- Random Forest classifier with the selected final parameters  

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier

# Final deployment model using the selected best configuration
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42
)

# Train on full dataset for deployment
final_model.fit(X, y)

# Save model
MODEL_DIR = PROJECT_ROOT / "models" / "spiral"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

final_model_path = MODEL_DIR / "spiral_model_combined.pkl"
joblib.dump(final_model, final_model_path)

print("Final deployment model saved to:", final_model_path)

## Final Model Evaluation
This section shows the performance of the selected Random Forest model using all features.

In [ ]:
# Final selected model evaluation (Random Forest with all 12 features)

rf_final = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42
)

rf_final.fit(X_train, y_train)
y_pred = rf_final.predict(X_test)

print("Final Model (Random Forest)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

## Overall Model Comparison Summary
This section summarizes the final model performance, including both test-set results and cross-validation metrics.

In [ ]:
# Overall Experiment Summary

# Final combined model comparison (Test + Cross-Validation)
spiral_model_comparison_summary = spiral_final_results_df.copy()

spiral_model_comparison_summary

In [ ]:
import joblib
import os
import copy

# Find best model by CV F1 — more reliable than test F1
best_model_name = cv_results_df.loc[
    cv_results_df['CV F1 Mean'].idxmax(), 'Model'
]

print(f'Best model by CV F1   : {best_model_name}')
print(f'CV F1 Mean            : {cv_results_df.loc[cv_results_df["CV F1 Mean"].idxmax(), "CV F1 Mean"]:.4f}')
print(f'Test Accuracy         : {results_df.loc[results_df["Model"]==best_model_name, "Accuracy"].values[0]*100:.2f}%')

# Save best model trained on full dataset
final_model = copy.deepcopy(models[best_model_name])
final_model.fit(X, y)

joblib.dump(final_model, MODEL_DIR / "spiral_model_combined.pkl")
print(f"Spiral model V2 saved — Best model: {best_model_name}")